# ComfyUI on Google Colab (Free GPU)
Run your custom ComfyUI setup from Hugging Face on a free Google Colab GPU.

### Instructions
1. Click **Runtime** > **Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Run the cell below to set up and start ComfyUI.
3. Wait for the **Cloudflare URL** (e.g., `https://random-name.trycloudflare.com`) to appear in the output. Click it to open ComfyUI.

In [ ]:
#@title Setup and Run ComfyUI
import os
import subprocess
import time
import re
import sys

# 0. Install System Dependencies
print('🔧 Installing system dependencies (ffmpeg, libgl1)...')
!apt-get update > /dev/null 2>&1
!apt-get install -y ffmpeg libgl1 libglib2.0-0 git git-lfs > /dev/null 2>&1

# 1. Install Git LFS and Clone Repository
print('🚀 Cloning repository from Hugging Face (including models)...')
!git lfs install
if not os.path.exists('/content/ComfyUI'):
    !git clone https://huggingface.co/spaces/anonymous4Chain/ComfyUI-GPU-Space /content/ComfyUI
else:
    print('Repository already exists, pulling latest changes...')
    %cd /content/ComfyUI
    !git pull

# --- FIX CUSTOM NODES MANUALLY ---
print('📦 Fixing broken custom nodes by re-cloning them with VERIFIED URLs...')
%cd /content/ComfyUI/custom_nodes
# Remove potentially broken folders
!rm -rf ComfyUI-Manager ComfyUI-VideoHelperSuite ComfyUI-MuseTalk ComfyUI-WanVideoWrapper ComfyUI_wav2lip ComfyUI_NTCosyVoice CosyVoice-ComfyUI TTS-Audio-Suite ComfyUI-LTXVideo ComfyUI_IPAdapter_plus

# Clone fresh versions (Verified URLs)
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!git clone https://github.com/chaojie/ComfyUI-MuseTalk.git
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
!git clone https://github.com/ShmuelRonen/ComfyUI_wav2lip.git
!git clone https://github.com/muxueChen/ComfyUI_NTCosyVoice.git
!git clone https://github.com/AIFSH/CosyVoice-ComfyUI.git
!git clone https://github.com/diodiogod/TTS-Audio-Suite.git
!git clone https://github.com/Lightricks/ComfyUI-LTXVideo.git
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git

%cd /content/ComfyUI

# 2. Install Dependencies
print('📦 Installing Python dependencies (this takes a few minutes)...')

# Force uninstall potentially conflicting packages
!pip uninstall -y ffmpeg ffmpeg-python webrtcvad > /dev/null 2>&1

# Fix for Python 3.12 webrtcvad error (CosyVoice)
print('🔧 Patching webrtcvad for Python 3.12...')
!pip install webrtcvad-wheels > /dev/null 2>&1

# Explicitly install heavy dependencies
print('📦 Installing specific node requirements (CosyVoice, LTX, VHS, MMPose)...')
!pip install opencv-python imageio-ffmpeg ninja onnxruntime-gpu > /dev/null 2>&1
!pip install diffusers transformers huggingface_hub > /dev/null 2>&1
!pip install hydra-core HyperPyYAML inflect librosa lightning matplotlib modelscope networkx omegaconf openai-whisper protobuf pydantic rich soundfile tensorboard wget gdown pyarrow jieba pypinyin pydub audiosegment srt > /dev/null 2>&1

# Robust installation for MM libraries (MuseTalk)
print('🔧 Installing MM libraries for MuseTalk...')
!pip install -U openmim > /dev/null 2>&1
!mim install mmengine > /dev/null 2>&1
!mim install "mmcv>=2.0.1" > /dev/null 2>&1
!mim install "mmdet>=3.1.0" > /dev/null 2>&1
!mim install "mmpose>=1.1.0" > /dev/null 2>&1

# Re-install ffmpeg-python specifically for MuseTalk
print('🔧 Fixing ffmpeg-python...')
!pip install ffmpeg-python > /dev/null 2>&1

# Attempt to install deepspeed (optional)
print('⚡ Installing DeepSpeed (optional)...')
!pip install deepspeed || echo 'DeepSpeed install failed, continuing...'

# Install generic requirements
print('📦 Installing generic project requirements...')
!pip install -r requirements.txt > /dev/null 2>&1

# Run the auto-installer for any leftover custom node requirements
import glob
req_files = glob.glob('custom_nodes/*/requirements.txt')
for f in req_files:
    print(f'Scanning: {f}')
    !pip install -r '{f}' > /dev/null 2>&1

# 3. Install Cloudflared for Public URL
print('🌐 Setting up Cloudflare Tunnel...')
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 4. Run ComfyUI
print('⚡ Starting ComfyUI...')

with open('comfyui.log', 'w') as log_file:
    comfy_process = subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'], stdout=log_file, stderr=subprocess.STDOUT)

with open('cloudflared.log', 'w') as cf_log:
    subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=cf_log, stderr=subprocess.STDOUT)

print('⏳ Waiting for Cloudflare URL to appear...')
url_found = False
retries = 0
while not url_found and retries < 60:
    time.sleep(2)
    retries += 1
    if os.path.exists('cloudflared.log'):
        with open('cloudflared.log', 'r') as f:
            content = f.read()
            match = re.search(r'https://[-a-zA-Z0-9]+[.]trycloudflare[.]com', content)
            if match:
                print('')
                print('SUCCESS: Public URL: ' + match.group(0))
                print('')
                url_found = True
    
    if comfy_process.poll() is not None:
        print('❌ ComfyUI process crashed! Printing logs:')
        with open('comfyui.log', 'r') as f:
            print(f.read())
        break

if url_found:
    print('📜 Streaming ComfyUI Logs (Errors will appear here)...')
    !tail -f comfyui.log
else:
    print('❌ Timed out waiting for URL. Checking cloudflared logs:')
    !cat cloudflared.log
